In [11]:
import pandas as pd
import numpy as np
from sklearn.ensemble import AdaBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn.model_selection import LeaveOneGroupOut, cross_val_score

Keep only labels for stress and neutral

In [12]:
features = pd.read_csv("../Data/wesad_eda/feature_dataframe.csv")
neutral = features[features['label'] == 1]
stress = features[features['label'] == 2]
feature_binary = pd.concat([neutral, stress])
feature_binary.columns

Index(['label', 'subject_id', 'phasic_mean', 'peaks_count',
       'phasic_std_rise_time', 'tonic_mean', 'tonic_max'],
      dtype='object')

In [13]:
# feature_binary = feature_binary.dropna()
feature_binary = feature_binary.drop(columns = ['phasic_std_rise_time'])
y = feature_binary[['label']].values.ravel()
X = feature_binary.iloc[:, 2:]

Reduced dimension is from keeping only rows with select labels.

In [14]:
print(X.shape)
print(y.shape)

(1769, 4)
(1769,)


Run test-train split

In [15]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42) 

Run Adaboost

In [16]:
groups = feature_binary['subject_id']
model = AdaBoostClassifier(n_estimators=50, learning_rate=1)
logo = LeaveOneGroupOut()
# Run the Cross-Validation
scores = cross_val_score(
    model, 
    X, 
    y, 
    groups=groups, 
    cv=logo, 
    scoring='accuracy'
)

# 4. Review the results
print(f"Number of subjects (folds): {len(scores)}")
print(f"Accuracy for each subject: {scores}")
print(f"Average Accuracy across all unseen subjects: {scores.mean():.4f}")

Number of subjects (folds): 15
Accuracy for each subject: [0.83471074 0.70833333 0.89830508 0.68067227 0.975      0.875
 0.75409836 0.87610619 0.71304348 1.         0.94067797 1.
 0.92307692 0.94067797 0.94827586]
Average Accuracy across all unseen subjects: 0.8712


In [17]:
# 1. Fit the model on the full dataset
model.fit(X, y)

# 2. Now you can safely get feature importances
importances = pd.Series(model.feature_importances_, index=X.columns)
print(importances.sort_values(ascending=False).head(10))

tonic_mean     0.381384
tonic_max      0.293592
phasic_mean    0.287083
peaks_count    0.037941
dtype: float64
